In [16]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

df = pd.read_csv("customer_transactions.csv")

df = df.drop_duplicates()

df["purchase_outcome"] = df["purchase_outcome"].map({
    "yes": 1,
    "no": 0
})
df.dropna(subset=["purchase_outcome"], inplace=True)

X = df.drop("purchase_outcome", axis=1)
y = df["purchase_outcome"]

numeric_columns = X.select_dtypes(include=["int64", "float64"]).columns
categorical_columns = X.select_dtypes(include=["object"]).columns

preprocessor = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), numeric_columns),

    ("cat", Pipeline([
        ("missing", SimpleImputer(strategy="most_frequent")),
        ("encode", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_columns)
])

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", model)
])

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1-score :", f1_score(y_test, y_pred))

print("\n--- Class Distribution in y_test ---")
print(y_test.value_counts())

print("\n--- Class Distribution in y_pred ---")
print(pd.Series(y_pred).value_counts())

Accuracy : 0.765
Precision: 0.0
Recall   : 0.0
F1-score : 0.0

--- Class Distribution in y_test ---
purchase_outcome
0    157
1     43
Name: count, dtype: int64

--- Class Distribution in y_pred ---
0    196
1      4
Name: count, dtype: int64


In [17]:
from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression(
    solver='liblinear',
    class_weight='balanced',
    random_state=42
)

logistic_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", logistic_model)
])

logistic_pipeline.fit(X_train, y_train)

y_pred_lr = logistic_pipeline.predict(X_test)

print("--- Logistic Regression Metrics ---")
print("Accuracy :", accuracy_score(y_test, y_pred_lr))
print("Precision:", precision_score(y_test, y_pred_lr))
print("Recall   :", recall_score(y_test, y_pred_lr))
print("F1-score :", f1_score(y_test, y_pred_lr))

print("\n--- Class Distribution in y_pred (Logistic Regression) ---")
print(pd.Series(y_pred_lr).value_counts())

--- Logistic Regression Metrics ---
Accuracy : 0.555
Precision: 0.23863636363636365
Recall   : 0.4883720930232558
F1-score : 0.32061068702290074

--- Class Distribution in y_pred (Logistic Regression) ---
0    112
1     88
Name: count, dtype: int64


In [18]:
from imblearn.over_sampling import SMOTE

print("--- Class Distribution in y_train (Original) ---")
print(y_train.value_counts())

--- Class Distribution in y_train (Original) ---
purchase_outcome
0    627
1    173
Name: count, dtype: int64


In [19]:
from imblearn.pipeline import Pipeline as ImbPipeline

model_smote = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

pipeline_smote = ImbPipeline([
    ("preprocessing", preprocessor),
    ('smote', SMOTE(random_state=42)),
    ("model", model_smote)
])

pipeline_smote.fit(X_train, y_train)

y_pred_smote = pipeline_smote.predict(X_test)

print("--- Random Forest Metrics (with SMOTE) ---")
print("Accuracy :", accuracy_score(y_test, y_pred_smote))
print("Precision:", precision_score(y_test, y_pred_smote))
print("Recall   :", recall_score(y_test, y_pred_smote))
print("F1-score :", f1_score(y_test, y_pred_smote))

print("\n--- Class Distribution in y_pred (Random Forest with SMOTE) ---")
print(pd.Series(y_pred_smote).value_counts())

--- Random Forest Metrics (with SMOTE) ---
Accuracy : 0.76
Precision: 0.0
Recall   : 0.0
F1-score : 0.0

--- Class Distribution in y_pred (Random Forest with SMOTE) ---
0    195
1      5
Name: count, dtype: int64


In [26]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

param_grid = {
    'model__n_estimators': [50, 100, 200],
    'model__max_depth': [5, 10, 20, None],
    'smote__k_neighbors': [3, 5, 7]
}

grid_search_recall = GridSearchCV(
    estimator=pipeline_smote,
    param_grid=param_grid,
    scoring='recall',
    cv=5,
    n_jobs=-1,
    verbose=1
)

print("Starting GridSearchCV with recall scoring...")
grid_search_recall.fit(X_train, y_train)
print("GridSearchCV completed.")

print("\nBest parameters found (optimized for recall):", grid_search_recall.best_params_)
print("Best Recall score found (cross-validation):", grid_search_recall.best_score_)

best_model_recall = grid_search_recall.best_estimator_

y_pred_tuned_recall = best_model_recall.predict(X_test)

print("\n--- Random Forest Metrics (with SMOTE and Hyperparameter Tuning, optimized for Recall) ---")
print("Accuracy :", accuracy_score(y_test, y_pred_tuned_recall))
print("Precision:", precision_score(y_test, y_pred_tuned_recall))
print("Recall   :", recall_score(y_test, y_pred_tuned_recall))
print("F1-score :", f1_score(y_test, y_pred_tuned_recall))

print("\n--- Class Distribution in y_pred (Random Forest with SMOTE and Hyperparameter Tuning, optimized for Recall) ---")
print(pd.Series(y_pred_tuned_recall).value_counts())

Starting GridSearchCV with recall scoring...
Fitting 5 folds for each of 36 candidates, totalling 180 fits
GridSearchCV completed.

Best parameters found (optimized for recall): {'model__max_depth': 10, 'model__n_estimators': 50, 'smote__k_neighbors': 7}
Best Recall score found (cross-validation): 0.04588235294117647

--- Random Forest Metrics (with SMOTE and Hyperparameter Tuning, optimized for Recall) ---
Accuracy : 0.79
Precision: 0.6
Recall   : 0.06976744186046512
F1-score : 0.125

--- Class Distribution in y_pred (Random Forest with SMOTE and Hyperparameter Tuning, optimized for Recall) ---
0    195
1      5
Name: count, dtype: int64
